In [1]:
library(Seurat)
library(harmony)

library(ggplot2)
library(ggplotify)
library(ggsci)
library(scatterpie)
library(patchwork)

library(ComplexHeatmap)
library(grid)
library(circlize)

library(dplyr)

source('../R_function/knn_function.R')
source('../R_function/calculate_function.R')
source('../R_function/ST_plot.R')
source('../R_function/plot_function.R')

Attaching SeuratObject

Loading required package: Rcpp

Loading required package: grid

ComplexHeatmap version 2.14.0
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github page: https://github.com/jokergoo/ComplexHeatmap
Documentation: http://jokergoo.github.io/ComplexHeatmap-reference

If you use it in published research, please cite either one:
- Gu, Z. Complex Heatmap Visualization. iMeta 2022.
- Gu, Z. Complex heatmaps reveal patterns and correlations in multidimensional 
    genomic data. Bioinformatics 2016.


The new InteractiveComplexHeatmap package can directly export static 
complex heatmaps into an interactive Shiny app with zero effort. Have a try!

This message can be suppressed by:
  suppressPackageStartupMessages(library(ComplexHeatmap))


circlize version 0.4.16
CRAN page: https://cran.r-project.org/package=circlize
Github page: https://github.com/jokergoo/circlize
Documentation: https://jokergoo.github.io/circlize_book/book/

If you use it in publi

In [ ]:
library(org.Mm.eg.db)  # 小鼠注释数据库
library(AnnotationDbi) # 注释数据操作核心包
library(dplyr) 
library(GO.db)
GO <- as.list(GOTERM)

In [47]:
OG_gene <- read.csv('/mnt/gandan/huangzhi/brain/02.Process_file/01.Multi_Reference/OrthoFinder/Results_Jul01/MyDirectory/OG_adj_1028.csv',sep=',',row.names = 1)
mouse_OG_gene <- OG_gene[OG_gene$species=='Mouse',]
rownames(mouse_OG_gene) <- mouse_OG_gene$self_gene

In [4]:
go2gene <- AnnotationDbi::mapIds(
  org.Mm.eg.db,                # 小鼠注释库
  keys = keys(org.Mm.eg.db, keytype = "GO"),  # 所有GO term
  keytype = "GO",              # 检索键类型：GO term
  column = "SYMBOL",        # 要提取的基因ID类型：Entrez ID
  multiVals = "list"           # 一个GO对应多个基因时，返回列表
)
go2ont <- AnnotationDbi::mapIds(
  org.Mm.eg.db,
  keys = keys(org.Mm.eg.db, keytype = "GO"),
  keytype = "GO",
  column = "ONTOLOGY",
  multiVals = "first"
)
go_gene_df <- dplyr::bind_rows(lapply(names(go2gene), function(x){
    df_tmp <- data.frame(gene=as.character(unlist(go2gene[x])))
    df_tmp$GO <- x
    df_tmp$ont <- go2ont[x]
    df_tmp$term <- GO[[x]]@Term
    return(df_tmp)}))

'select()' returned 1:many mapping between keys and columns



In [54]:
go_gene_df$OG <- mouse_OG_gene[go_gene_df$gene,'OG']
go_og_df <- go_gene_df[!is.na(go_gene_df),]
go_og_df <- aggregate(list('gene'=go_gene_df$gene),by=list('GO'=go_gene_df$GO,'ont'=go_gene_df$ont,'term'=go_gene_df$term,'OG'=go_gene_df$OG),function(x){return(paste0(x,collapse = ','))})

In [56]:
write.csv(go_og_df,'Analysis/OG_mous_GO_annotation.csv')

In [59]:
mmu_kegg <- download_KEGG(species = "mmu")

Reading KEGG annotation online: "https://rest.kegg.jp/link/mmu/pathway"...

Reading KEGG annotation online: "https://rest.kegg.jp/list/pathway/mmu"...



In [68]:
saveRDS(mmu_kegg,'Analysis/mmu_kegg.rds')

In [75]:
kegg_gene_df <- as.data.frame(mmu_kegg$KEGGPATHID2EXTID)
colnames(kegg_gene_df) <- c("pathwayid", "geneid")
KEGG_term <- mmu_kegg$KEGGPATHID2NAME
rownames(KEGG_term) <- KEGG_term$from
kegg_gene_df$term <- gsub(' - Mus musculus \\(house mouse\\)','',KEGG_term[kegg_gene_df$pathwayid,'to'])

In [85]:
kegg_gene_df$gene <- mapIds(
  org.Mm.eg.db,
  keys = kegg_gene_df$geneid,
  keytype = "ENTREZID",
  column = "SYMBOL",
  multiVals = "first"
)
kegg_gene_df$OG <- mouse_OG_gene[kegg_gene_df$gene,'OG']
kegg_og_df <- kegg_gene_df[!is.na(kegg_gene_df),]
kegg_og_df <- aggregate(list('gene'=kegg_og_df$gene),by=list('pathwayid'=kegg_og_df$pathwayid,'term'=kegg_og_df$term,'OG'=kegg_og_df$OG),function(x){return(paste0(x,collapse = ','))})

'select()' returned 1:1 mapping between keys and columns



In [87]:
write.csv(kegg_og_df,'Analysis/OG_mous_KEGG_annotation.csv')

In [9]:
length(unique(go_og_df$term))
length(unique(go_og_df$GO))

[1] 18945

[1] 18945